# Paper Figure Console

This notebook is the paper-side console for the repository. It loads saved artifacts from `results/`, calls plotting utilities from `src.plotting`, previews the figures, and exports them for the paper.

## Environment Check

Run this cell first to locate the project root, expose the source package, and confirm the local notebook environment.

In [1]:
import sys
from pathlib import Path

import pandas as pd
import plotly
from IPython.display import HTML, display


def find_project_root(start_path: Path) -> Path:
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "src" / "model").exists() and (candidate / "experiment.ipynb").exists():
            return candidate
    raise RuntimeError("Could not locate the project root from the current working directory.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.plotting import generate_default_paper_figures, load_paper_figure_data, save_figures

pd.set_option("display.max_colwidth", 160)

versions_df = pd.DataFrame(
    [
        {"package": "python", "version": sys.version.split()[0]},
        {"package": "pandas", "version": pd.__version__},
        {"package": "plotly", "version": plotly.__version__},
    ]
)
display(versions_df)
print(f"Project root: {PROJECT_ROOT}")
print(f"Python executable: {sys.executable}")


,package,version
0,python,3.13.12
1,pandas,2.3.3
2,plotly,6.7.0


Project root: D:\5329_assignment2
Python executable: d:\5329_assignment2\.venv\Scripts\python.exe


## Configuration

Set the experiment name, the seeds to load, the seed used for qualitative plots, and the case index to visualize.

In [2]:
EXPERIMENT_NAME = "distilbert_sst2_multiseed"
SEEDS = [42, 43, 44, 45]
INSPECT_SEED = 42
QUALITATIVE_CASE_INDEX = 519
TOP_N_TOKENS = 8
FIGURE_OUTPUT_DIR = PROJECT_ROOT / "paper" / "figures"

print(f"Experiment name: {EXPERIMENT_NAME}")
print(f"Seeds: {SEEDS}")
print(f"Inspect seed: {INSPECT_SEED}")
print(f"Qualitative case index: {QUALITATIVE_CASE_INDEX}")
print(f"Figure output directory: {FIGURE_OUTPUT_DIR}")


Experiment name: distilbert_sst2_multiseed
Seeds: [42, 43, 44, 45]
Inspect seed: 42
Qualitative case index: 519
Figure output directory: D:\5329_assignment2\paper\figures


## Load Artifact Tables

This cell loads validation metrics, ranking-similarity summaries, deletion summaries, and the seed-specific files needed for qualitative plots.

In [3]:
data = load_paper_figure_data(
    project_root=PROJECT_ROOT,
    experiment_name=EXPERIMENT_NAME,
    seeds=SEEDS,
    inspect_seed=INSPECT_SEED,
)

display(data.validation_metrics)
display(data.validation_metrics_aggregate)
display(data.ranking_similarity_aggregate)
display(
    data.deletion_aggregate[
        data.deletion_aggregate["k_setting_type"].eq("top_k")
        & data.deletion_aggregate["actual_k"].isin([1, 2, 3])
    ].sort_values(["method", "actual_k"]).reset_index(drop=True)
)


,seed,accuracy,f1,loss,num_examples
0,42,0.912844,0.915179,0.318047,872
1,43,0.904817,0.908891,0.347401,872
2,44,0.911697,0.910776,0.275913,872
3,45,0.900229,0.902793,0.376587,872


,metric,mean,std,min,max
0,accuracy,0.907397,0.005153,0.900229,0.912844
1,f1,0.909410,0.004450,0.902793,0.915179
2,loss,0.329487,0.037217,0.275913,0.376587
3,num_examples,872.000000,0.000000,872.000000,872.000000


,method_a,method_b,num_seeds,mean_ranked_token_count_mean,mean_ranked_token_count_std,mean_spearman_rank_correlation_mean,mean_spearman_rank_correlation_std
0,attention,grad_x_input,4,23.163991,0.0,0.223906,0.012353
1,attention,loo,4,23.163991,0.0,0.176133,0.001357
2,grad_x_input,loo,4,23.163991,0.0,0.087480,0.007430


,method,k_setting_type,k_setting_value,actual_k,num_seeds,mean_confidence_drop_mean,mean_confidence_drop_std,mean_gold_label_prob_drop_mean,mean_gold_label_prob_drop_std,mean_target_prob_drop_mean,mean_target_prob_drop_std,flip_rate_mean,flip_rate_std,becomes_incorrect_rate_mean,becomes_incorrect_rate_std
0,attention,top_k,1.0,1,4,0.013754,0.004596,0.092031,0.007870,0.127832,0.006503,0.139048,0.006968,0.185206,0.008401
1,attention,top_k,2.0,2,4,0.019817,0.007828,0.134749,0.013092,0.182335,0.010026,0.197534,0.010526,0.228211,0.012387
2,attention,top_k,3.0,3,4,0.027289,0.011785,0.171993,0.013488,0.221861,0.006142,0.239655,0.011359,0.265805,0.013515
3,grad_x_input,top_k,1.0,1,4,0.007260,0.003058,0.046376,0.004853,0.069135,0.004991,0.081422,0.006817,0.138475,0.007966
4,grad_x_input,top_k,2.0,2,4,0.013752,0.004300,0.077764,0.007201,0.110331,0.014467,0.122706,0.013793,0.167718,0.012885
5,grad_x_input,top_k,3.0,3,4,0.016181,0.004617,0.102101,0.004068,0.141395,0.013763,0.158046,0.009865,0.197989,0.000575
6,loo,top_k,1.0,1,4,0.038286,0.012586,0.180419,0.011437,0.284003,0.014936,0.302179,0.014216,0.272076,0.009566
7,loo,top_k,2.0,2,4,0.042175,0.012019,0.260971,0.015516,0.382536,0.010480,0.405677,0.008951,0.356651,0.017692
8,loo,top_k,3.0,3,4,0.045807,0.009741,0.318389,0.021440,0.452249,0.011298,0.473851,0.012743,0.413793,0.020838
9,random,top_k,1.0,1,4,0.002952,0.002048,0.014419,0.001791,0.028504,0.004973,0.038303,0.002060,0.107970,0.004341


## Generate And Preview Figures

The notebook below behaves like a console: it asks `src.plotting` for the default figure bundle and previews every figure inline.

In [4]:
def display_plotly_figure(fig):
    try:
        fig.show()
    except ValueError as exc:
        if "nbformat" not in str(exc):
            raise
        display(HTML(fig.to_html(include_plotlyjs="cdn")))


figures = generate_default_paper_figures(
    data,
    qualitative_case_index=QUALITATIVE_CASE_INDEX,
    top_n_tokens=TOP_N_TOKENS,
)

for figure_name, figure in figures.items():
    print(f"=== {figure_name} ===")
    display_plotly_figure(figure)


=== fig01_experiment_workflow ===


=== fig02_validation_metrics ===


=== fig03_ranking_similarity_heatmap ===


=== fig04_confidence_drop_topk ===


=== fig05_flip_rate_topk ===


=== figA1_confidence_drop_ratio ===


=== figA2_flip_rate_ratio ===


=== fig06_case_rankings_seed42_idx519 ===


## Export Figures

Run this cell to save every figure to `paper/figures/` as HTML, and as PNG when the local image export backend is available.

In [5]:
manifest_df = save_figures(figures, FIGURE_OUTPUT_DIR, save_png=True, save_html=True)
display(manifest_df)


,figure_name,html_path,png_path,png_saved,png_error
0,fig01_experiment_workflow,D:\5329_assignment2\paper\figures\fig01_experiment_workflow.html,D:\5329_assignment2\paper\figures\fig01_experiment_workflow.png,True,
1,fig02_validation_metrics,D:\5329_assignment2\paper\figures\fig02_validation_metrics.html,D:\5329_assignment2\paper\figures\fig02_validation_metrics.png,True,
2,fig03_ranking_similarity_heatmap,D:\5329_assignment2\paper\figures\fig03_ranking_similarity_heatmap.html,D:\5329_assignment2\paper\figures\fig03_ranking_similarity_heatmap.png,True,
3,fig04_confidence_drop_topk,D:\5329_assignment2\paper\figures\fig04_confidence_drop_topk.html,D:\5329_assignment2\paper\figures\fig04_confidence_drop_topk.png,True,
4,fig05_flip_rate_topk,D:\5329_assignment2\paper\figures\fig05_flip_rate_topk.html,D:\5329_assignment2\paper\figures\fig05_flip_rate_topk.png,True,
5,figA1_confidence_drop_ratio,D:\5329_assignment2\paper\figures\figA1_confidence_drop_ratio.html,D:\5329_assignment2\paper\figures\figA1_confidence_drop_ratio.png,True,
6,figA2_flip_rate_ratio,D:\5329_assignment2\paper\figures\figA2_flip_rate_ratio.html,D:\5329_assignment2\paper\figures\figA2_flip_rate_ratio.png,True,
7,fig06_case_rankings_seed42_idx519,D:\5329_assignment2\paper\figures\fig06_case_rankings_seed42_idx519.html,D:\5329_assignment2\paper\figures\fig06_case_rankings_seed42_idx519.png,True,
